In [ ]:
# Environment Setup
import sys
import os
sys.path.append(os.path.abspath('../..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# QUANTUM-FORGE imports
from core.mathematical_engine.pricing_models import PricingModels
from core.market_microstructure.order_flow import OrderFlowAnalyzer
from core.ai_ml_intelligence.feature_engineering import FeatureEngineer
from components.analytics.performance_analytics import PerformanceAnalytics

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("  Environment setup complete")

## 1. Data Loading & Preprocessing

Load market data and prepare it for alpha signal generation.

In [ ]:
class AlphaDataLoader:
    """Load and preprocess data for alpha research."""
    
    def __init__(self):
        self.data = None
        self.returns = None
        self.features = None
    
    def load_market_data(self, symbols, start_date, end_date):
        """Load market data for specified symbols."""
        # Placeholder for actual data loading
        dates = pd.date_range(start_date, end_date, freq='1min')
        
        data = {}
        for symbol in symbols:
            # Simulate market data
            np.random.seed(hash(symbol) % 2**32)
            prices = 100 * np.exp(np.cumsum(np.random.randn(len(dates)) * 0.001))
            volumes = np.random.lognormal(10, 1, len(dates))
            
            data[symbol] = pd.DataFrame({
                'open': prices * (1 + np.random.randn(len(dates)) * 0.0001),
                'high': prices * (1 + np.abs(np.random.randn(len(dates))) * 0.0002),
                'low': prices * (1 - np.abs(np.random.randn(len(dates))) * 0.0002),
                'close': prices,
                'volume': volumes
            }, index=dates)
        
        self.data = data
        return data
    
    def calculate_returns(self, periods=[1, 5, 10, 20]):
        """Calculate returns over multiple periods."""
        returns_data = {}
        
        for symbol, df in self.data.items():
            returns_data[symbol] = pd.DataFrame(index=df.index)
            for period in periods:
                returns_data[symbol][f'returns_{period}'] = df['close'].pct_change(period)
        
        self.returns = returns_data
        return returns_data
    
    def get_summary_statistics(self):
        """Get summary statistics of loaded data."""
        summary = {}
        for symbol, df in self.data.items():
            summary[symbol] = {
                'observations': len(df),
                'mean_price': df['close'].mean(),
                'volatility': df['close'].pct_change().std() * np.sqrt(252 * 390),
                'mean_volume': df['volume'].mean()
            }
        return pd.DataFrame(summary).T

# Initialize and load data
loader = AlphaDataLoader()
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
data = loader.load_market_data(symbols, '2024-01-01', '2024-12-31')
returns = loader.calculate_returns()

print("  Data loaded successfully")
loader.get_summary_statistics()

## 2. Alpha Signal Generation

Generate various alpha signals from market data using technical, fundamental, and microstructure factors.

In [ ]:
class AlphaSignalGenerator:
    """Generate alpha signals from market data."""
    
    def __init__(self, data):
        self.data = data
        self.signals = {}
    
    def momentum_signal(self, symbol, lookback=20):
        """Generate momentum alpha signal."""
        df = self.data[symbol]
        returns = df['close'].pct_change(lookback)
        signal = (returns - returns.mean()) / returns.std()
        return signal.fillna(0)
    
    def mean_reversion_signal(self, symbol, lookback=20, std_dev=2):
        """Generate mean reversion alpha signal."""
        df = self.data[symbol]
        ma = df['close'].rolling(lookback).mean()
        std = df['close'].rolling(lookback).std()
        z_score = (df['close'] - ma) / std
        signal = -z_score  # Negative because we expect reversion
        return signal.fillna(0)
    
    def volume_signal(self, symbol, lookback=20):
        """Generate volume-based alpha signal."""
        df = self.data[symbol]
        vol_ma = df['volume'].rolling(lookback).mean()
        vol_ratio = df['volume'] / vol_ma
        price_change = df['close'].pct_change()
        signal = vol_ratio * np.sign(price_change)
        signal = (signal - signal.mean()) / signal.std()
        return signal.fillna(0)
    
    def volatility_signal(self, symbol, lookback=20):
        """Generate volatility-based alpha signal."""
        df = self.data[symbol]
        returns = df['close'].pct_change()
        volatility = returns.rolling(lookback).std()
        vol_ma = volatility.rolling(lookback).mean()
        signal = (volatility - vol_ma) / vol_ma
        signal = (signal - signal.mean()) / signal.std()
        return signal.fillna(0)
    
    def rsi_signal(self, symbol, period=14):
        """Generate RSI-based alpha signal."""
        df = self.data[symbol]
        delta = df['close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        # Transform to signal: oversold = positive, overbought = negative
        signal = (50 - rsi) / 50
        return signal.fillna(0)
    
    def macd_signal(self, symbol, fast=12, slow=26, signal_period=9):
        """Generate MACD-based alpha signal."""
        df = self.data[symbol]
        exp1 = df['close'].ewm(span=fast).mean()
        exp2 = df['close'].ewm(span=slow).mean()
        macd = exp1 - exp2
        signal_line = macd.ewm(span=signal_period).mean()
        signal = macd - signal_line
        signal = (signal - signal.mean()) / signal.std()
        return signal.fillna(0)
    
    def generate_all_signals(self, symbol):
        """Generate all alpha signals for a symbol."""
        signals = pd.DataFrame(index=self.data[symbol].index)
        signals['momentum'] = self.momentum_signal(symbol)
        signals['mean_reversion'] = self.mean_reversion_signal(symbol)
        signals['volume'] = self.volume_signal(symbol)
        signals['volatility'] = self.volatility_signal(symbol)
        signals['rsi'] = self.rsi_signal(symbol)
        signals['macd'] = self.macd_signal(symbol)
        
        self.signals[symbol] = signals
        return signals

# Generate signals for all symbols
signal_gen = AlphaSignalGenerator(data)
all_signals = {symbol: signal_gen.generate_all_signals(symbol) for symbol in symbols}

print("  Alpha signals generated")
print(f"\nSignals for {symbols[0]}:")
all_signals[symbols[0]].tail()

## 3. Alpha Performance Evaluation

Evaluate the performance of individual alpha signals.

In [ ]:
class AlphaEvaluator:
    """Evaluate alpha signal performance."""
    
    def __init__(self, signals, returns):
        self.signals = signals
        self.returns = returns
    
    def calculate_ic(self, symbol, signal_name, forward_period=1):
        """Calculate Information Coefficient (IC)."""
        signal = self.signals[symbol][signal_name]
        forward_returns = self.returns[symbol][f'returns_{forward_period}'].shift(-forward_period)
        
        # Calculate correlation
        valid_data = pd.DataFrame({'signal': signal, 'returns': forward_returns}).dropna()
        if len(valid_data) > 0:
            ic = valid_data['signal'].corr(valid_data['returns'])
            return ic
        return 0
    
    def calculate_ic_stats(self, symbol, signal_name, forward_period=1, rolling_window=1000):
        """Calculate rolling IC statistics."""
        signal = self.signals[symbol][signal_name]
        forward_returns = self.returns[symbol][f'returns_{forward_period}'].shift(-forward_period)
        
        # Rolling IC
        rolling_ic = pd.Series(index=signal.index, dtype=float)
        for i in range(rolling_window, len(signal)):
            window_signal = signal.iloc[i-rolling_window:i]
            window_returns = forward_returns.iloc[i-rolling_window:i]
            valid = pd.DataFrame({'s': window_signal, 'r': window_returns}).dropna()
            if len(valid) > 10:
                rolling_ic.iloc[i] = valid['s'].corr(valid['r'])
        
        return {
            'mean_ic': rolling_ic.mean(),
            'std_ic': rolling_ic.std(),
            'ir': rolling_ic.mean() / rolling_ic.std() if rolling_ic.std() > 0 else 0,
            'rolling_ic': rolling_ic
        }
    
    def evaluate_all_signals(self, symbol, forward_period=1):
        """Evaluate all signals for a symbol."""
        results = {}
        for signal_name in self.signals[symbol].columns:
            ic = self.calculate_ic(symbol, signal_name, forward_period)
            ic_stats = self.calculate_ic_stats(symbol, signal_name, forward_period)
            results[signal_name] = {
                'IC': ic,
                'Mean IC': ic_stats['mean_ic'],
                'IC Std': ic_stats['std_ic'],
                'IC IR': ic_stats['ir']
            }
        return pd.DataFrame(results).T

# Evaluate signals
evaluator = AlphaEvaluator(all_signals, returns)
evaluation_results = {}

for symbol in symbols:
    evaluation_results[symbol] = evaluator.evaluate_all_signals(symbol)
    print(f"\n{symbol} Signal Evaluation:")
    print(evaluation_results[symbol].round(4))

## 4. Alpha Decay Analysis

Analyze how alpha signals decay over different time horizons.

In [ ]:
def analyze_alpha_decay(symbol, signal_name, max_horizon=20):
    """Analyze alpha decay across multiple horizons."""
    decay_results = []
    
    for horizon in range(1, max_horizon + 1):
        signal = all_signals[symbol][signal_name]
        forward_returns = returns[symbol]['returns_1'].shift(-horizon)
        
        valid_data = pd.DataFrame({'signal': signal, 'returns': forward_returns}).dropna()
        if len(valid_data) > 0:
            ic = valid_data['signal'].corr(valid_data['returns'])
            decay_results.append({'horizon': horizon, 'ic': ic})
    
    return pd.DataFrame(decay_results)

# Analyze decay for top signals
decay_analysis = {}
for symbol in symbols[:2]:  # Analyze first 2 symbols
    decay_analysis[symbol] = {}
    for signal_name in ['momentum', 'mean_reversion', 'rsi']:
        decay_analysis[symbol][signal_name] = analyze_alpha_decay(symbol, signal_name)

# Visualize decay
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[f"{sym} Alpha Decay" for sym in symbols[:2]]
)

for idx, symbol in enumerate(symbols[:2]):
    for signal_name in ['momentum', 'mean_reversion', 'rsi']:
        df = decay_analysis[symbol][signal_name]
        fig.add_trace(
            go.Scatter(x=df['horizon'], y=df['ic'], name=signal_name, mode='lines+markers'),
            row=1, col=idx+1
        )

fig.update_xaxes(title_text="Forecast Horizon (periods)")
fig.update_yaxes(title_text="Information Coefficient")
fig.update_layout(height=400, title_text="Alpha Signal Decay Analysis")
fig.show()

print("  Alpha decay analysis complete")

## 5. Signal Combination

Combine multiple alpha signals using various weighting schemes.

In [ ]:
class SignalCombiner:
    """Combine multiple alpha signals."""
    
    def __init__(self, signals, returns):
        self.signals = signals
        self.returns = returns
    
    def equal_weight_combination(self, symbol):
        """Equal weight signal combination."""
        signals_df = self.signals[symbol]
        combined = signals_df.mean(axis=1)
        return combined
    
    def ic_weighted_combination(self, symbol, lookback=1000):
        """IC-weighted signal combination."""
        signals_df = self.signals[symbol]
        forward_returns = self.returns[symbol]['returns_1'].shift(-1)
        
        # Calculate rolling ICs
        weights = pd.DataFrame(index=signals_df.index, columns=signals_df.columns)
        
        for i in range(lookback, len(signals_df)):
            window_returns = forward_returns.iloc[i-lookback:i]
            ics = []
            for col in signals_df.columns:
                window_signal = signals_df[col].iloc[i-lookback:i]
                valid = pd.DataFrame({'s': window_signal, 'r': window_returns}).dropna()
                if len(valid) > 10:
                    ic = valid['s'].corr(valid['r'])
                    ics.append(max(ic, 0))  # Only positive ICs
                else:
                    ics.append(0)
            
            # Normalize weights
            total = sum(ics)
            if total > 0:
                weights.iloc[i] = [ic / total for ic in ics]
            else:
                weights.iloc[i] = [1/len(ics)] * len(ics)
        
        # Combine signals
        combined = (signals_df * weights).sum(axis=1)
        return combined
    
    def pca_combination(self, symbol, n_components=1):
        """PCA-based signal combination."""
        signals_df = self.signals[symbol].fillna(0)
        
        # Standardize signals
        scaler = StandardScaler()
        signals_scaled = scaler.fit_transform(signals_df)
        
        # Apply PCA
        pca = PCA(n_components=n_components)
        combined = pca.fit_transform(signals_scaled)
        
        return pd.Series(combined.flatten(), index=signals_df.index)
    
    def evaluate_combination(self, symbol, combined_signal, forward_period=1):
        """Evaluate combined signal performance."""
        forward_returns = self.returns[symbol][f'returns_{forward_period}'].shift(-forward_period)
        valid_data = pd.DataFrame({'signal': combined_signal, 'returns': forward_returns}).dropna()
        
        if len(valid_data) > 0:
            ic = valid_data['signal'].corr(valid_data['returns'])
            return ic
        return 0

# Combine signals
combiner = SignalCombiner(all_signals, returns)

combination_results = {}
for symbol in symbols:
    ew_signal = combiner.equal_weight_combination(symbol)
    ic_signal = combiner.ic_weighted_combination(symbol)
    pca_signal = combiner.pca_combination(symbol)
    
    combination_results[symbol] = {
        'Equal Weight': combiner.evaluate_combination(symbol, ew_signal),
        'IC Weighted': combiner.evaluate_combination(symbol, ic_signal),
        'PCA': combiner.evaluate_combination(symbol, pca_signal)
    }

combination_df = pd.DataFrame(combination_results).T
print("\nSignal Combination Performance (IC):")
print(combination_df.round(4))

## 6. Statistical Significance Testing

Test the statistical significance of alpha signals.

In [ ]:
def test_signal_significance(symbol, signal_name, n_simulations=1000):
    """Test signal significance using permutation test."""
    signal = all_signals[symbol][signal_name]
    forward_returns = returns[symbol]['returns_1'].shift(-1)
    
    valid_data = pd.DataFrame({'signal': signal, 'returns': forward_returns}).dropna()
    actual_ic = valid_data['signal'].corr(valid_data['returns'])
    
    # Permutation test
    permuted_ics = []
    for _ in range(n_simulations):
        permuted_returns = valid_data['returns'].sample(frac=1).values
        permuted_ic = stats.pearsonr(valid_data['signal'].values, permuted_returns)[0]
        permuted_ics.append(permuted_ic)
    
    # Calculate p-value
    p_value = np.mean(np.abs(permuted_ics) >= np.abs(actual_ic))
    
    return {
        'signal': signal_name,
        'actual_ic': actual_ic,
        'p_value': p_value,
        'significant': p_value < 0.05
    }

# Test significance for all signals
significance_results = {}
for symbol in symbols[:2]:  # Test first 2 symbols
    significance_results[symbol] = []
    for signal_name in all_signals[symbol].columns:
        result = test_signal_significance(symbol, signal_name)
        significance_results[symbol].append(result)
    
    print(f"\n{symbol} Significance Tests:")
    sig_df = pd.DataFrame(significance_results[symbol])
    print(sig_df.round(4))

## 7. Real-Time Alpha Monitoring Dashboard

Create an interactive dashboard for monitoring alpha performance in real-time.

In [ ]:
def create_alpha_dashboard(symbol):
    """Create interactive alpha monitoring dashboard."""
    
    # Prepare data
    signals_df = all_signals[symbol]
    prices = data[symbol]['close']
    
    # Create subplots
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=[
            'Price Action', 'Combined Alpha Signal',
            'Individual Signals', 'Signal Correlation',
            'Rolling IC', 'Signal Distribution'
        ],
        specs=[
            [{'secondary_y': False}, {'secondary_y': False}],
            [{'secondary_y': False}, {'secondary_y': False}],
            [{'secondary_y': False}, {'secondary_y': False}]
        ]
    )
    
    # 1. Price action
    fig.add_trace(
        go.Scatter(x=prices.index, y=prices.values, name='Price', line=dict(color='blue')),
        row=1, col=1
    )
    
    # 2. Combined signal
    combined = combiner.equal_weight_combination(symbol)
    fig.add_trace(
        go.Scatter(x=combined.index, y=combined.values, name='Combined Alpha', 
                   line=dict(color='green')),
        row=1, col=2
    )
    
    # 3. Individual signals
    for col in signals_df.columns[:3]:  # Show top 3 signals
        fig.add_trace(
            go.Scatter(x=signals_df.index, y=signals_df[col], name=col, mode='lines'),
            row=2, col=1
        )
    
    # 4. Signal correlation heatmap
    corr_matrix = signals_df.corr()
    fig.add_trace(
        go.Heatmap(z=corr_matrix.values, x=corr_matrix.columns, y=corr_matrix.columns,
                   colorscale='RdBu', zmid=0),
        row=2, col=2
    )
    
    # 5. Rolling IC
    for signal_name in signals_df.columns[:3]:
        ic_stats = evaluator.calculate_ic_stats(symbol, signal_name)
        rolling_ic = ic_stats['rolling_ic'].dropna()
        fig.add_trace(
            go.Scatter(x=rolling_ic.index, y=rolling_ic.values, name=f"{signal_name} IC"),
            row=3, col=1
        )
    
    # 6. Signal distribution
    for col in signals_df.columns[:3]:
        fig.add_trace(
            go.Histogram(x=signals_df[col].dropna(), name=col, opacity=0.6),
            row=3, col=2
        )
    
    # Update layout
    fig.update_layout(
        height=1200,
        title_text=f"{symbol} Alpha Research Dashboard",
        showlegend=True
    )
    
    return fig

# Create dashboard for first symbol
dashboard = create_alpha_dashboard(symbols[0])
dashboard.show()

print("  Alpha monitoring dashboard created")

## 8. Key Findings & Recommendations

### Summary of Results
- **Top Performing Signals**: Identify signals with highest IC and IC IR
- **Signal Decay**: Understand optimal holding periods
- **Combination Strategy**: Evaluate best signal combination approach
- **Statistical Significance**: Validate signal robustness

### Next Steps
1. Implement selected signals in paper trading
2. Monitor real-time performance
3. Optimize signal parameters
4. Develop risk management overlay
5. Scale to production environment

In [ ]:
# Export results for production
def export_alpha_signals(symbol, output_path='alpha_signals.csv'):
    """Export alpha signals for production use."""
    signals_df = all_signals[symbol].copy()
    combined = combiner.ic_weighted_combination(symbol)
    signals_df['combined_alpha'] = combined
    signals_df.to_csv(output_path)
    print(f"  Alpha signals exported to {output_path}")

# Example export
# export_alpha_signals(symbols[0])

print("\n" + "="*60)
print("Alpha Strategy Research Complete")
print("="*60)